# NLP Lab (CSE405R02): Exam Notebook

**How every exercise is organised (run it step by step)**
- Each exercise is split into **steps**. One step = **one function**, defined in a cell and **called right below it** in the same cell, so you see its output before moving on.
- Run the cells **top to bottom**. The result of one step is passed to the next step as a function argument / return value.
- The first code cell of an exercise only holds the **sample data** (text, probabilities, grammar). Change it to try your own input.
- Every exercise is **self-contained** (own imports and data), so any one can be run alone.

**Ground rules used here**
- Pure Python 3, **standard library only** (`re`, `math`, `random`, `collections`, `json`, `urllib`). No `pip install` needed.
- Layout follows the lab record: **Aim, Idea, Procedure, Steps (functions), Result**, plus a few **viva points**.

| Ex | Topic | Manual asks for | Done here with |
|----|-------|-----------------|----------------|
| 1 | Tokenize, POS, NER, parse tree, ambiguity | NLTK | regex + small rules + recursive parser |
| 2 | Collocation: t-test, chi-square | NLTK corpus | `math`, `Counter` |
| 3 | Naive Bayes word-sense disambiguation | none | `math`, `Counter` |
| 4 | Hindle and Rooth PP attachment | none | `math`, `Counter` |
| 5 | HMM forward / backward | NumPy | plain lists |
| 6 | Viterbi | none | plain dicts |
| 7 | PCFG + CYK inside probability | NLTK `Tree` | dicts + own tree printer |
| 8 | BoW / TF-IDF classification | TensorFlow | own vectorizers + logistic regression |
| 9 | word2vec (CBOW, Skip-gram) | gensim | own tiny neural net |
| 10 | Text generator | Keras LSTM | n-gram model (+ LSTM gate demo) |
| 11 | RNN machine translation | Keras LSTM | own RNN encoder-decoder, hand-written backprop |
| 12 | LLM chatbot | `anthropic` package | `urllib` call + offline fallback bot |

---
# Exercise 1: Tokenize, POS-tag, NER, parse tree, ambiguity

**Aim:** Tokenize and tag text, identify named entities, display a parse tree, and find the ambiguity of a sentence using parse trees.

**Idea**
- *Tokenizing* splits text into sentences and words. *Stop words* ("the", "a") carry little meaning.
- *Stemming* chops suffixes (`studies` to `studi`); *lemmatization* returns a real word (`studies` to `study`).
- *POS tagging* gives each word a grammatical tag (NN noun, VB verb, DT determiner, NNP proper noun, CD number).
- *NER* groups proper nouns into entities (PERSON, ORGANIZATION, GPE, DATE).
- *Parsing* builds a tree from a grammar. A sentence is **ambiguous** if it has more than one parse tree.

**Steps (one function each):**
1. `tokenize` 2. `remove_stopwords` 3. `stem` / `lemmatize` 4. `pos_tag` 5. `ne_chunk` 6. `show_parse_tree` 7. `check_ambiguity`

*NLTK's trained models are replaced by small hand-written rules and word lists.*

In [ ]:
# Sample data (change me)
import re

text = ("Sundar Pichai is the CEO of Google. He visited Chennai in March 2025. "
        "The sun moves eastward.")

STOPWORDS = {"a", "an", "the", "is", "are", "was", "of", "in", "on", "to", "and", "he", "she", "it"}
IRREGULAR = {"ate": "eat", "eaten": "eat", "went": "go", "was": "be", "mice": "mouse"}
LEXICON = {"the": "DT", "a": "DT", "an": "DT", "is": "VBZ", "are": "VBP", "of": "IN", "in": "IN",
           "he": "PRP", "she": "PRP", "i": "PRP", "it": "PRP", "visited": "VBD", "moves": "VBZ",
           "eastward": "RB", "ceo": "NN", "sun": "NN", "with": "IN", "saw": "VBD"}
MONTHS = {"january", "february", "march", "april", "may", "june", "july", "august",
          "september", "october", "november", "december"}
GAZETTEER = {"google": "ORGANIZATION", "microsoft": "ORGANIZATION", "sastra": "ORGANIZATION",
             "chennai": "GPE", "thanjavur": "GPE", "india": "GPE"}

In [ ]:
# Step 1: sentence and word tokenization
def tokenize(text):
    # split after . ! ? when a capital letter follows
    sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z])", text)
    # a word/number, or a single punctuation mark
    words = re.findall(r"[A-Za-z0-9']+|[^\sA-Za-z0-9]", text)
    return sentences, words


sentences, words = tokenize(text)
print("Sentences:")
for s in sentences:
    print("  ", s)
print("\nWords:", words)

In [ ]:
# Step 2: remove stop words
def remove_stopwords(words):
    return [w for w in words if w.isalnum() and w.lower() not in STOPWORDS]


print("Without stop words:", remove_stopwords(words))

In [ ]:
# Step 3: stemming and lemmatization
def stem(word):
    """Crude stemmer: chop a known suffix (Porter does the same with far more rules)."""
    for suffix in ("ing", "ed", "es", "en", "s"):
        if word.endswith(suffix) and len(word) - len(suffix) >= 3:
            return word[:-len(suffix)]
    return word


def lemmatize(word):
    """The lemma is always a real word: irregular list first, then simple endings."""
    if word in IRREGULAR:
        return IRREGULAR[word]
    if word.endswith("ies"):
        return word[:-3] + "y"
    if word.endswith("s") and not word.endswith("ss"):
        return word[:-1]
    if word.endswith("ing") and len(word) > 5:
        return word[:-3]
    if word.endswith("ed") and len(word) > 4:
        return word[:-2]
    return word


for w in ["eating", "eats", "eaten", "studies", "moves", "visited"]:
    print(f"{w:10s} stem = {stem(w):8s} lemma = {lemmatize(w)}")

In [ ]:
# Step 4: POS tagging = lexicon lookup, then suffix / shape rules for unknown words
def pos_tag(tokens):
    tagged = []
    for w in tokens:
        lw = w.lower()
        if lw in LEXICON:
            tag = LEXICON[lw]
        elif w.isdigit():
            tag = "CD"
        elif not w.isalnum():
            tag = "."                      # punctuation
        elif w[0].isupper():
            tag = "NNP"                    # capitalised word = proper noun
        elif lw.endswith("ing"):
            tag = "VBG"
        elif lw.endswith("ed"):
            tag = "VBD"
        elif lw.endswith("ly"):
            tag = "RB"
        elif lw.endswith("s"):
            tag = "NNS"
        else:
            tag = "NN"
        tagged.append((w, tag))
    return tagged


tagged = pos_tag(words)
print(tagged)

In [ ]:
# Step 5: Named entity recognition = group runs of NNP and label them
def ne_chunk(tagged):
    entities, i = [], 0
    while i < len(tagged):
        if tagged[i][1] != "NNP":
            i += 1
            continue
        j = i                                          # extend over consecutive NNP words
        while j + 1 < len(tagged) and tagged[j + 1][1] == "NNP":
            j += 1
        name = [w for w, _ in tagged[i:j + 1]]
        if name[0].lower() in MONTHS:                  # "March 2025" is a DATE
            if j + 1 < len(tagged) and tagged[j + 1][1] == "CD":
                j += 1
                name.append(tagged[j][0])
            label = "DATE"
        else:
            label = GAZETTEER.get(name[-1].lower(), "PERSON")   # unknown name defaults to PERSON
        entities.append((" ".join(name), label))
        i = j + 1
    return entities


for name, label in ne_chunk(tagged):
    print(f"{label:13s} {name}")

In [ ]:
# Step 6a: the parser. A symbol that is not a key of the grammar is a word.
def parse(symbol, words, i, grammar):
    """Yield (tree, next_position) for every way `symbol` can match words starting at i."""
    if symbol not in grammar:                          # terminal word
        if i < len(words) and words[i] == symbol:
            yield symbol, i + 1
        return
    for rhs in grammar[symbol]:                        # try each rule  symbol -> rhs
        for children, j in parse_seq(rhs, words, i, grammar):
            yield (symbol, children), j


def parse_seq(symbols, words, i, grammar):
    """Match a sequence of symbols one after another."""
    if not symbols:
        yield [], i
        return
    for tree, j in parse(symbols[0], words, i, grammar):
        for rest, k in parse_seq(symbols[1:], words, j, grammar):
            yield [tree] + rest, k


def all_parses(sentence, grammar):
    words = sentence.lower().split()
    return [t for t, end in parse("S", words, 0, grammar) if end == len(words)]


print("parser defined")

In [ ]:
# Step 6b: display a parse tree
def bracket(tree):
    if isinstance(tree, str):
        return tree
    label, kids = tree
    return "(" + label + " " + " ".join(bracket(k) for k in kids) + ")"


def draw(tree, indent=""):
    """Print the tree sideways: children are indented under their parent."""
    if isinstance(tree, str):
        print(indent + tree)
        return
    label, kids = tree
    if len(kids) == 1 and isinstance(kids[0], str):
        print(f"{indent}{label} -> {kids[0]}")
    else:
        print(indent + label)
        for k in kids:
            draw(k, indent + "    ")


def show_parse_tree(sentence, grammar):
    for tree in all_parses(sentence, grammar):
        print(bracket(tree), "\n")
        draw(tree)


grammar = {
    "S":  [["NP", "VP"]],
    "NP": [["A", "NN"]],
    "VP": [["V", "NN"]],
    "A":  [["the"]],
    "NN": [["sun"], ["eastward"]],
    "V":  [["moves"]],
}
show_parse_tree("The sun moves eastward", grammar)

In [ ]:
# Step 7: ambiguity = more than one parse tree for the same sentence
def check_ambiguity(sentence, grammar):
    trees = all_parses(sentence, grammar)
    print(f"'{sentence}' has {len(trees)} parse tree(s)\n")
    for n, tree in enumerate(trees, 1):
        vp = tree[1][1]                                      # the VP node under S
        meaning = ("PP attaches to the VERB: I used the telescope to see the man"
                   if len(vp[1]) == 3 else
                   "PP attaches to the NOUN: the man has the telescope")
        print(f"Parse {n}: {meaning}")
        draw(tree, "  ")
        print()
    print("The sentence is AMBIGUOUS" if len(trees) > 1 else "The sentence is unambiguous")


amb_grammar = {
    "S":   [["NP", "VP"]],
    "NP":  [["Det", "N"], ["Det", "N", "PP"], ["i"]],
    "VP":  [["V", "NP"], ["V", "NP", "PP"]],
    "PP":  [["P", "NP"]],
    "Det": [["the"]],
    "N":   [["man"], ["telescope"]],
    "V":   [["saw"]],
    "P":   [["with"]],
}
check_ambiguity("I saw the man with the telescope", amb_grammar)          # <- change me

**Result:** The text was tokenized and tagged, named entities were identified, parse trees were displayed, and the ambiguity of the sentence was shown by its two parse trees.

**Viva points**
- Stemming is rule-based chopping and may give non-words; lemmatization uses a dictionary and gives valid words.
- Ambiguity here is *PP-attachment*: the same words, two tree shapes, two meanings.
- Real NLTK uses trained taggers (perceptron / maxent); ours uses a lexicon plus suffix rules.

---
# Exercise 2: Collocation with the t-test and chi-square test

**Aim:** Check whether a given sequence of words is a collocation using the t-test and the chi-square test.

**Idea:** A *collocation* is a word pair that occurs together more often than chance (`strong tea`, `heavy rain`).
We compare how often the pair is **observed** with how often it would appear if the two words were **independent**.

- Null hypothesis H0: the words are independent, so `P(w1 w2) = P(w1) x P(w2)`.
- **t-test:**  $t = \dfrac{\bar{x} - \mu}{\sqrt{s^2 / N}}$
  where $\bar{x}$ = observed bigram probability $C(w_1 w_2)/N$, $\mu = P(w_1)P(w_2)$, $s^2 \approx \bar{x}$ (Bernoulli), $N$ = number of bigrams.
  Reject H0 (so it **is** a collocation) if $t >$ 1.645 (alpha = 0.05, one-tailed; use 2.576 for the stricter alpha = 0.005).
- **Chi-square:**  $\chi^2 = \sum_{i,j} \dfrac{(O_{ij} - E_{ij})^2}{E_{ij}}$ on the 2x2 table
  `[[w1 w2, w1 not-w2], [not-w1 w2, neither]]`. Reject H0 if $\chi^2 >$ 3.841 (alpha = 0.05, 1 degree of freedom).

**Steps (one function each):** 1. `count_words` 2. `t_score` / `t_test` 3. `chi_square` / `chi_test` 4. `rank_bigrams` (bonus, with PMI)

In [ ]:
# Sample data (change me)
import math
import re
from collections import Counter

# A small corpus (the phrases we care about repeat, as they would in a big corpus)
corpus = """
She drinks strong tea every morning. He prefers strong tea with milk. Strong tea keeps me awake.
My uncle makes strong tea for guests. Nobody likes weak tea. The cafe serves green tea and hot coffee.
A strong wind blew across the field. The strong coffee was too bitter. A strong team won the match.
There was heavy rain last night. Heavy rain flooded the street. They cancelled the trip because of heavy rain.
The forecast says heavy rain tomorrow. We stayed at home during the heavy rain. Light rain is pleasant.
The heavy bag fell on the floor. Heavy traffic delayed the bus. She carried a heavy box upstairs.
Machine learning is a branch of artificial intelligence. Machine learning needs a lot of data.
Students study machine learning in the third year. Machine learning models learn from examples.
Natural language processing helps computers read text. Natural language processing is fun.
Natural language processing powers chatbots. Many students enjoy natural language processing.
Fast food is not always healthy. They ate fast food after the match. Fast food shops are crowded at night.
The fast train arrived early. The old man walked to the market. The market was open on Sunday.
Tea and coffee are popular drinks. The rain stopped in the morning. The students walked to the library.
"""

STOPWORDS = {"a", "an", "the", "is", "are", "was", "were", "of", "in", "on", "to", "and", "at", "for", "from",
             "by", "with", "me", "my", "she", "he", "they", "we", "it", "there", "after", "because", "during",
             "across", "not", "too", "last", "every", "many", "always", "into", "lot", "i", "has", "that", "this"}

T_CRIT = 1.645        # t-test, alpha = 0.05 one-tailed (2.576 for alpha = 0.005)
CHI_CRIT = 3.841      # chi-square, alpha = 0.05, 1 d.o.f.
MIN_COUNT = 3         # chi-square is unreliable for very rare pairs, so we do not judge them

pairs = [("strong", "tea"), ("heavy", "rain"), ("machine", "learning"), ("natural", "language"),
         ("fast", "food"), ("strong", "wind"), ("weak", "tea"), ("tea", "coffee")]

In [ ]:
# Step 1: tokenize and count unigrams and bigrams
def count_words(corpus):
    tokens = re.findall(r"[a-z]+", corpus.lower())            # lowercase, punctuation removed
    return {"N": len(tokens),
            "unigram": Counter(tokens),
            "bigram": Counter(zip(tokens, tokens[1:]))}


counts = count_words(corpus)
print("tokens:", counts["N"], "| distinct words:", len(counts["unigram"]), "| distinct bigrams:", len(counts["bigram"]))

In [ ]:
# Step 2: t-test
def t_score(w1, w2, counts):
    N, unigram, bigram = counts["N"], counts["unigram"], counts["bigram"]
    c12 = bigram[(w1, w2)]
    if c12 == 0:
        return 0.0
    x_bar = c12 / N                                   # observed mean
    mu = (unigram[w1] / N) * (unigram[w2] / N)        # expected mean if independent
    return (x_bar - mu) / math.sqrt(x_bar / N)        # s^2 is about x_bar


def verdict(score, critical, w1, w2, counts):
    if counts["bigram"][(w1, w2)] < MIN_COUNT:
        return "are not collocated! (seen fewer than 3 times)"
    return "are collocated!" if score > critical else "are not collocated!"


def t_test(pairs, counts):
    for w1, w2 in pairs:
        t = t_score(w1, w2, counts)
        print(f"t-value: {t:6.2f}   Words: {w1} and {w2} {verdict(t, T_CRIT, w1, w2, counts)}")


print("t-test:")
t_test(pairs, counts)

In [ ]:
# Step 3: chi-square test
def chi_square(w1, w2, counts):
    N, unigram, bigram = counts["N"], counts["unigram"], counts["bigram"]
    o11 = bigram[(w1, w2)]                            # w1 w2
    o12 = unigram[w1] - o11                           # w1 followed by something else
    o21 = unigram[w2] - o11                           # w2 preceded by something else
    o22 = N - o11 - o12 - o21                         # neither
    observed = [[o11, o12], [o21, o22]]
    row = [sum(r) for r in observed]
    col = [observed[0][j] + observed[1][j] for j in range(2)]
    chi = 0.0
    for i in range(2):
        for j in range(2):
            expected = row[i] * col[j] / N
            if expected:
                chi += (observed[i][j] - expected) ** 2 / expected
    return chi


def chi_test(pairs, counts):
    for w1, w2 in pairs:
        chi = chi_square(w1, w2, counts)
        print(f"chi-value: {chi:8.2f}   Words: {w1} and {w2} {verdict(chi, CHI_CRIT, w1, w2, counts)}")


print("chi^2-test:")
chi_test(pairs, counts)

In [ ]:
# Step 4 (bonus): rank every non-stop-word bigram, plus PMI = log2( P(w1 w2) / (P(w1) P(w2)) )
def pmi(w1, w2, counts):
    N, unigram, bigram = counts["N"], counts["unigram"], counts["bigram"]
    return math.log2((bigram[(w1, w2)] / N) / ((unigram[w1] / N) * (unigram[w2] / N)))


def rank_bigrams(counts, top=8):
    bigram = counts["bigram"]
    candidates = [b for b, c in bigram.items() if c >= 2 and b[0] not in STOPWORDS and b[1] not in STOPWORDS]
    print(f"{'bigram':25s} {'count':>5s} {'t':>7s} {'chi2':>9s} {'PMI':>6s}")
    for b in sorted(candidates, key=lambda b: -t_score(*b, counts))[:top]:
        print(f"{' '.join(b):25s} {bigram[b]:5d} {t_score(*b, counts):7.2f} "
              f"{chi_square(*b, counts):9.2f} {pmi(*b, counts):6.2f}")


rank_bigrams(counts)

**Result:** The t-test and the chi-square test identified `strong tea`, `heavy rain`, `machine learning`, `natural language` and `fast food` as collocations, and rejected random pairs such as `tea coffee`.

**Viva points**
- t-test uses the *mean/variance* of the bigram; chi-square uses a 2x2 *contingency table* and does not assume normality.
- A pair seen only once has t of about 1, so it can never pass; chi-square wrongly flags very rare pairs, so we skip pairs seen fewer than 3 times.
- PMI favours rare pairs; t-score favours frequent pairs. That is why both are shown.

---
# Exercise 3: Naive Bayes word-sense disambiguation (WSD)

**Aim:** Find the sense of an ambiguous word (`chair`) from its context using a decision rule based on Naive Bayes.

**Idea:** Pick the sense $s_k$ that maximises $P(s_k)\prod_j P(v_j \mid s_k)$ over the context words $v_j$ (the "naive" part:
context words are independent given the sense). We add logs so that tiny products do not underflow:

$score(s_k) = \log P(s_k) + \sum_{v_j \in c} \log P(v_j \mid s_k)$, choose $\arg\max_{s_k} score(s_k)$

- Training: $P(v_j \mid s_k) = C(v_j, s_k) / C(s_k)$ and $P(s_k) = C(s_k) / C(w)$.
- **Add-one smoothing** is used so an unseen context word does not give `log(0)`.

**Steps (one function each):** 1. `context_words` 2. `train_model` 3. `p_word` 4. `disambiguate` 5. `test_sentences`

In [ ]:
# Sample data (change me)
import math
import re
from collections import Counter

TARGET = "chair"
STOP = {"the", "a", "an", "of", "on", "in", "to", "and", "was", "is", "for", "with", "he", "she", "his", "her", "by", "at"}

# Training set: sentences labelled with the sense of "chair"
train = {
    "furniture": [
        "she sat on the wooden chair near the table",
        "the chair and the desk were in the office",
        "he bought a comfortable chair with soft cushion",
        "move the chair close to the dining table",
        "the old chair has a broken leg",
        "a leather chair stood beside the sofa",
    ],
    "position": [
        "the chair of the committee announced the meeting",
        "he was appointed chair of the department",
        "the chair presided over the board meeting",
        "she became chair of the university senate",
        "the chair resigned from the position yesterday",
        "the chair proposed a vote on the new policy",
    ],
}

tests = ["the chair of the board spoke first",
         "she sat on the old chair near the table",
         "please move the chair beside the desk"]

In [ ]:
# Step 1: extract the context words of a sentence
def context_words(sentence):
    """Lowercase words, minus stop words and the target word itself."""
    return [w for w in re.findall(r"[a-z]+", sentence.lower()) if w not in STOP and w != TARGET]


print(context_words("She sat on the wooden chair near the table"))

In [ ]:
# Step 2: training = count senses and (word, sense) pairs, then compute the priors P(s_k)
def train_model(train):
    sense_count = {s: len(sentences) for s, sentences in train.items()}              # C(s_k)
    total = sum(sense_count.values())                                                # C(w)
    word_count = {s: Counter(w for sent in sentences for w in context_words(sent))   # C(v_j, s_k)
                  for s, sentences in train.items()}
    vocab = {w for counts in word_count.values() for w in counts}
    prior = {s: sense_count[s] / total for s in train}                               # P(s_k)
    return {"prior": prior, "word_count": word_count, "vocab": vocab}


model = train_model(train)
print("Priors:", model["prior"])
print("Vocabulary size:", len(model["vocab"]))

In [ ]:
# Step 3: likelihood P(v_j | s_k) with add-one smoothing
def p_word(word, sense, model):
    word_count, vocab = model["word_count"], model["vocab"]
    return (word_count[sense][word] + 1) / (sum(word_count[sense].values()) + len(vocab))


print("P(table | furniture) =", round(p_word("table", "furniture", model), 4))
print("P(table | position)  =", round(p_word("table", "position", model), 4))

In [ ]:
# Step 4: disambiguation = score every sense, pick the largest
def disambiguate(sentence, model):
    context = context_words(sentence)
    scores = {s: math.log(model["prior"][s]) + sum(math.log(p_word(w, s, model)) for w in context)
              for s in model["prior"]}
    return max(scores, key=scores.get), scores


best, scores = disambiguate("the chair of the board spoke first", model)
print(scores, "->", best)

In [ ]:
# Step 5: test sentences
def test_sentences(sentences, model):
    for sentence in sentences:
        best, scores = disambiguate(sentence, model)
        print(f"\nSentence: {sentence}")
        print({s: round(v, 4) for s, v in scores.items()})
        print(f"The sense of 'Chair' is {best}.", scores[best])


test_sentences(tests, model)

**Result:** The Naive Bayes decision rule chose the most likely sense of `chair` for each test sentence from the training set.

**Viva points**
- Naive assumption: context words are independent given the sense. It is false for language, but the argmax is often still right.
- Without smoothing, one unseen word makes the whole product zero.
- Log scores are negative; the **larger (closer to 0)** score wins.

---
# Exercise 4: Hindle and Rooth algorithm (PP-attachment ambiguity)

**Aim:** Decide whether a prepositional phrase attaches to the **verb** or to the **noun**.

**Idea:** In `saw the man [with the telescope]` the PP could modify *saw* (verb attachment) or *man* (noun attachment).
Hindle and Rooth compare how strongly the preposition `p` is attracted to the verb `v` versus the noun `n`:

$\lambda(v,n,p) = \log_2 \dfrac{P(VA_p = 1 \mid v)\; P(NA_p = 0 \mid n)}{P(NA_p = 1 \mid n)}$

with $P(VA_p = 1 \mid v) = C(v,p)/C(v)$ and $P(NA_p = 1 \mid n) = C(n,p)/C(n)$.
If $\lambda > 0$ choose **verb** attachment, otherwise **noun** attachment.

*(The manual prints `P(NA=0 | v)`. The noun `n` is the correct conditioning variable, and it is what the code uses.)*

**Steps (one function each):** 1. `count_attachments` 2. `prob` 3. `attach` 4. `test_examples` 5. `try_your_own`

In [ ]:
# Sample data (change me)
import math
from collections import Counter

# Training data: (verb, noun, preposition, attachment) with V = verb attach, N = noun attach
data = [
    ("saw", "man", "with", "N"), ("saw", "star", "with", "V"), ("saw", "girl", "with", "V"),
    ("hit", "ball", "with", "V"), ("hit", "man", "with", "V"), ("hit", "ball", "with", "V"),
    ("bought", "book", "with", "N"), ("bought", "book", "with", "N"), ("bought", "car", "with", "V"),
    ("ate", "pizza", "with", "V"), ("ate", "pizza", "with", "N"), ("ate", "salad", "with", "V"),
    ("read", "book", "with", "N"), ("read", "story", "with", "V"), ("opened", "door", "with", "V"),
    ("opened", "box", "with", "V"), ("found", "ball", "with", "N"), ("found", "ball", "with", "N"),
    ("found", "key", "with", "N"), ("kicked", "ball", "with", "V"),
]

examples = [("hit", "ball", "with"), ("found", "ball", "with"), ("saw", "man", "with"), ("opened", "door", "with")]

In [ ]:
# Step 1: count verb+prep and noun+prep attachments in the corpus
def count_attachments(data):
    count_v, count_vp = Counter(), Counter()       # C(v),  C(v, p)  (verb attachments)
    count_n, count_np = Counter(), Counter()       # C(n),  C(n, p)  (noun attachments)
    for v, n, p, attachment in data:
        count_v[v] += 1
        count_n[n] += 1
        if attachment == "V":
            count_vp[(v, p)] += 1
        else:
            count_np[(n, p)] += 1
    return {"v": count_v, "vp": count_vp, "n": count_n, "np": count_np}


counts = count_attachments(data)
print("C(hit) =", counts["v"]["hit"], "| C(hit, with) =", counts["vp"][("hit", "with")])
print("C(ball) =", counts["n"]["ball"], "| C(ball, with) =", counts["np"][("ball", "with")])

In [ ]:
# Step 2: smoothed probability  (count + alpha) / (total + 2 alpha), so unseen words never give 0 or divide by zero
def prob(count, total, alpha=0.5):
    return (count + alpha) / (total + 2 * alpha)


print("P(VA_with = 1 | hit)  =", round(prob(counts["vp"][("hit", "with")], counts["v"]["hit"]), 4))
print("P(NA_with = 1 | ball) =", round(prob(counts["np"][("ball", "with")], counts["n"]["ball"]), 4))

In [ ]:
# Step 3: the decision rule lambda(v, n, p)
def attach(v, n, p, counts):
    p_va = prob(counts["vp"][(v, p)], counts["v"][v])         # P(VA_p = 1 | v)
    p_na = prob(counts["np"][(n, p)], counts["n"][n])         # P(NA_p = 1 | n)
    lam = math.log2(p_va * (1 - p_na) / p_na)
    return lam, ("Verb" if lam > 0 else "Noun")


print(attach("hit", "ball", "with", counts))

In [ ]:
# Step 4: test the examples
def test_examples(examples, counts):
    for v, n, p in examples:
        lam, result = attach(v, n, p, counts)
        print(f"{v} the {n} {p} ...: lambda = {lam:+.3f}  ->  {result} attachment")


test_examples(examples, counts)

In [ ]:
# Step 5: try your own (press Enter to accept the default in brackets)
def try_your_own(counts):
    v = input("Verb [hit]: ").strip().lower() or "hit"
    n = input("Noun [ball]: ").strip().lower() or "ball"
    p = input("Preposition [with]: ").strip().lower() or "with"
    lam, result = attach(v, n, p, counts)
    print(f"lambda = {lam:+.3f}")
    print(result)


try_your_own(counts)

**Result:** The Hindle and Rooth algorithm was implemented; for each (verb, noun, preposition) it printed whether the PP attaches to the verb or the noun.

**Viva points**
- It uses *lexical preference*: some verbs love a preposition (`opened ... with`), some nouns do (`man with`).
- $\lambda > 0$ means the verb is the more likely attachment site; the sign of $\lambda$ is the decision rule.
- It models one preposition at a time and only the first PP when two share a preposition.

---
# Exercise 5: Hidden Markov Model, forward and backward procedures

**Aim:** Use the forward and backward procedures of an HMM to find the probability of an observation (word) sequence.

**Idea:** An HMM has hidden states we cannot see and observations we can. It is defined by
$\pi_i$ (start probabilities), $a_{ij}$ (state transitions) and $b_j(o)$ (emissions).
Instead of summing over all $N^T$ state paths, dynamic programming does it in $O(N^2 T)$.

| | Forward | Backward |
|---|---|---|
| Init | $\alpha_1(j) = \pi_j\, b_j(o_1)$ | $\beta_T(i) = 1$ |
| Recursion | $\alpha_t(j) = \sum_i \alpha_{t-1}(i)\, a_{ij}\, b_j(o_t)$ | $\beta_t(i) = \sum_j a_{ij}\, b_j(o_{t+1})\, \beta_{t+1}(j)$ |
| End | $P(O) = \sum_j \alpha_T(j)$ | $P(O) = \sum_j \pi_j\, b_j(o_1)\, \beta_1(j)$ |

Both must give the **same** $P(O)$. Also $\gamma_t(i) = \alpha_t(i)\beta_t(i) / P(O)$ is the probability of being in state $i$ at time $t$.

**Steps (one function each):** 1. `forward` 2. `backward` 3. `compare` 4. `gamma_values`

In [ ]:
# Sample data (change me)
states = ["R", "S"]                      # hidden states
symbols = ["A", "B"]                     # observable symbols
pi = [1.0, 0.0]                          # start probabilities
A = [[0.7, 0.3],                         # A[i][j] = P(next state j | state i)
     [0.4, 0.6]]
B = [[0.8, 0.2],                         # B[j][k] = P(symbol k | state j)
     [0.3, 0.7]]

obs_seq = "A A B".split()
O = [symbols.index(o) for o in obs_seq]  # observations as numbers
N, T = len(states), len(O)
print("Observation sequence:", obs_seq, "->", O)

In [ ]:
# Step 1: forward procedure
def forward(O, pi, A, B):
    N, T = len(pi), len(O)
    alpha = [[pi[j] * B[j][O[0]] for j in range(N)]]                        # initialization
    for t in range(1, T):                                                   # recursion
        alpha.append([sum(alpha[t - 1][i] * A[i][j] for i in range(N)) * B[j][O[t]]
                      for j in range(N)])
    return alpha, sum(alpha[-1])                                            # termination


alpha, p_forward = forward(O, pi, A, B)
for t, row in enumerate(alpha, 1):
    print(f"alpha at t={t}:", [round(v, 5) for v in row])
print(f"P(O | model) by forward procedure = {p_forward:.6f}")

In [ ]:
# Step 2: backward procedure
def backward(O, pi, A, B):
    N, T = len(pi), len(O)
    beta = [[1.0] * N for _ in range(T)]                                    # initialization
    for t in range(T - 2, -1, -1):                                          # recursion
        beta[t] = [sum(A[i][j] * B[j][O[t + 1]] * beta[t + 1][j] for j in range(N))
                   for i in range(N)]
    return beta, sum(pi[j] * B[j][O[0]] * beta[0][j] for j in range(N))     # termination


beta, p_backward = backward(O, pi, A, B)
for t, row in enumerate(beta, 1):
    print(f"beta  at t={t}:", [round(v, 5) for v in row])
print(f"P(O | model) by backward procedure = {p_backward:.6f}")

In [ ]:
# Step 3: both procedures must agree
def compare(p_forward, p_backward):
    print(f"forward = {p_forward:.6f}   backward = {p_backward:.6f}")
    print("They match" if abs(p_forward - p_backward) < 1e-12 else "MISMATCH: check the model")


compare(p_forward, p_backward)

In [ ]:
# Step 4: gamma = P(state at time t | whole observation sequence)
def gamma_values(alpha, beta, p_total, states):
    for t in range(len(alpha)):
        g = [alpha[t][i] * beta[t][i] / p_total for i in range(len(states))]
        print(f"  t={t + 1}: " + ", ".join(f"{s}={v:.4f}" for s, v in zip(states, g)))


gamma_values(alpha, beta, p_forward, states)

**Result:** The forward and backward procedures gave the same probability for the observation sequence, confirming both implementations.

**Viva points**
- Forward looks at the *past* up to time $t$; backward looks at the *future* after time $t$.
- $\alpha_t(i)\,\beta_t(i)$ combines both, which is the basis of Baum-Welch training.
- Complexity: $O(N^2 T)$ instead of $O(N^T)$ for brute force.

---
# Exercise 6: Viterbi algorithm

**Aim:** Find the most probable hidden-state (tag) sequence for an observation sequence, and its probability, using an HMM.

**Idea:** Viterbi is the forward algorithm with **max** instead of **sum**, plus back-pointers to recover the path.

1. Init: $\delta_1(j) = \pi_j\, b_j(o_1)$
2. Induction: $\delta_t(j) = \max_i \delta_{t-1}(i)\, a_{ij}\, b_j(o_t)$, and store $\psi_t(j) = \arg\max_i \dots$
3. End: best probability $= \max_j \delta_T(j)$; follow $\psi$ **backwards** to read out the best path.

**Steps (one function each):** 1. `initialize` 2. `induction` 3. `backtrack` 4. `viterbi` (all together) 5. `show_table`

In [ ]:
# Sample data (change me): the weather example from the manual
states = ("Rainy", "Sunny")
observations = ("Walk", "Shop", "Clean")
start_probability = {"Rainy": 0.6, "Sunny": 0.4}
transition_probability = {"Rainy": {"Rainy": 0.7, "Sunny": 0.3},
                          "Sunny": {"Rainy": 0.4, "Sunny": 0.6}}
emission_probability = {"Rainy": {"Walk": 0.1, "Shop": 0.4, "Clean": 0.5},
                        "Sunny": {"Walk": 0.6, "Shop": 0.3, "Clean": 0.1}}

In [ ]:
# Step 1: initialization  delta_1(j) = pi_j * b_j(o_1)
def initialize(obs, states, start_p, emit_p):
    return {s: start_p[s] * emit_p[s][obs[0]] for s in states}


delta_first = initialize(observations, states, start_probability, emission_probability)
print(delta_first)

In [ ]:
# Step 2: induction = for every time step and state keep the best previous state (back-pointer)
def induction(obs, states, trans_p, emit_p, first_delta):
    delta, back = [first_delta], [{}]
    for t in range(1, len(obs)):
        delta.append({})
        back.append({})
        for s in states:
            prob, prev = max((delta[t - 1][r] * trans_p[r][s] * emit_p[s][obs[t]], r) for r in states)
            delta[t][s], back[t][s] = prob, prev
    return delta, back


delta, back = induction(observations, states, transition_probability, emission_probability, delta_first)
for t, obs in enumerate(observations):
    print(f"t={t + 1} ({obs}): delta = {delta[t]}  back-pointers = {back[t]}")

In [ ]:
# Step 3: termination and backtracking
def backtrack(delta, back, states):
    best_prob, last = max((delta[-1][s], s) for s in states)
    path = [last]
    for t in range(len(delta) - 1, 0, -1):
        path.insert(0, back[t][path[0]])
    return round(best_prob, 6), path


print(backtrack(delta, back, states))

In [ ]:
# Step 4: all steps together as one function
def viterbi(obs, states, start_p, trans_p, emit_p):
    first = initialize(obs, states, start_p, emit_p)
    delta, back = induction(obs, states, trans_p, emit_p, first)
    return backtrack(delta, back, states), delta


(best_prob, best_path), delta = viterbi(observations, states, start_probability,
                                        transition_probability, emission_probability)
print("Best path:", best_path, "| probability:", best_prob)       # expected 0.01344, Sunny Rainy Rainy

In [ ]:
# Step 5: show the delta table
def show_table(obs, states, delta):
    print(f"{'':8s}" + "".join(f"{o:>12s}" for o in obs))
    for s in states:
        print(f"{s:8s}" + "".join(f"{delta[t][s]:12.5f}" for t in range(len(obs))))


show_table(observations, states, delta)

In [ ]:
# The same algorithm as a part-of-speech tagger: states = tags, observations = words
tags = ("DET", "NOUN", "VERB")
words = ("the", "dog", "barks")
start = {"DET": 0.8, "NOUN": 0.15, "VERB": 0.05}
trans = {"DET": {"DET": 0.01, "NOUN": 0.98, "VERB": 0.01},
         "NOUN": {"DET": 0.1, "NOUN": 0.2, "VERB": 0.7},
         "VERB": {"DET": 0.5, "NOUN": 0.4, "VERB": 0.1}}
emit = {"DET": {"the": 0.9, "dog": 0.0, "barks": 0.0},
        "NOUN": {"the": 0.0, "dog": 0.5, "barks": 0.1},
        "VERB": {"the": 0.0, "dog": 0.01, "barks": 0.6}}

(prob, best_tags), delta = viterbi(words, tags, start, trans, emit)
show_table(words, tags, delta)
print("\nBest tag sequence:", list(zip(words, best_tags)), "| probability:", prob)

**Result:** The Viterbi algorithm found the most likely tag sequence and its probability. For the weather example it gave `['Sunny', 'Rainy', 'Rainy']` with probability 0.01344.

**Viva points**
- Forward **sums** over all paths (gives $P(O)$); Viterbi takes the **max** (gives the single best path).
- Back-pointers are needed because the max only stores probabilities, not which state led to them.
- With long sentences multiply probabilities in **log space** to avoid underflow.

---
# Exercise 7: PCFG and the CYK algorithm (inside probability)

**Aim:** Build a Probabilistic Context-Free Grammar (PCFG) and find the inside probability of a sentence with the CYK algorithm.

**Idea:** A PCFG gives every rule a probability (the rules for one left side add up to 1).
CYK works on a grammar in **Chomsky Normal Form** (rules are `A -> B C` or `A -> word`) and fills a table over spans:

- Base: $\pi[i,i,A] = P(A \to w_i)$
- Recursive: $\pi[i,j,A] \mathrel{+}= P(A \to B\,C)\;\pi[i,m,B]\;\pi[m+1,j,C]$ for every split point $m$

**Inside probability** $\beta_A(i,j)$ *sums* over all parses of words $i..j$ from $A$. $P(\text{sentence}) = \beta_S(1,n)$.
Replacing the sum by a **max** gives the Viterbi (best) parse.

**Steps (one function each):** 1. `fill_base_case` 2. `fill_spans` 3. `cyk` (both together) 4. `build_tree` 5. `run`

In [ ]:
# Sample data (change me): the PCFG in Chomsky Normal Form
from collections import defaultdict

lexical = {("Det", "the"): 1.0, ("N", "cat"): 0.5, ("N", "dog"): 0.5,       # A -> word
           ("V", "chased"): 1.0, ("P", "with"): 1.0}
binary = {("S", "NP", "VP"): 1.0,                                          # A -> B C
          ("NP", "Det", "N"): 0.7, ("NP", "NP", "PP"): 0.3,
          ("VP", "V", "NP"): 0.7, ("VP", "VP", "PP"): 0.3,
          ("PP", "P", "NP"): 1.0}
NONTERMINALS = sorted({a for a, _ in lexical} | {a for a, _, _ in binary})
print("Non-terminals:", NONTERMINALS)

In [ ]:
# Step 1: base case  pi[i, i, A] = P(A -> w_i)
def fill_base_case(words, inside, best):
    for i, w in enumerate(words):
        for A in NONTERMINALS:
            p = lexical.get((A, w), 0.0)
            if p:
                inside[i, i, A] = best[i, i, A] = p


inside, best, back = defaultdict(float), defaultdict(float), {}
fill_base_case(["the", "cat", "chased", "the", "dog"], inside, best)
print({k: v for k, v in inside.items()})

In [ ]:
# Step 2: recursive case, span by span
def fill_spans(n, inside, best, back):
    for span in range(2, n + 1):
        for i in range(n - span + 1):
            j = i + span - 1
            for m in range(i, j):                                    # split point
                for (A, B, C), p in binary.items():
                    left, right = inside[i, m, B], inside[m + 1, j, C]
                    if left and right:
                        inside[i, j, A] += p * left * right          # inside: SUM over parses
                        candidate = p * best[i, m, B] * best[m + 1, j, C]
                        if candidate > best[i, j, A]:                # best parse: MAX
                            best[i, j, A] = candidate
                            back[i, j, A] = (m, B, C)


fill_spans(5, inside, best, back)
print("cells filled:", len(inside))

In [ ]:
# Step 3: the whole CYK in one function
def cyk(words):
    inside, best, back = defaultdict(float), defaultdict(float), {}
    fill_base_case(words, inside, best)
    fill_spans(len(words), inside, best, back)
    return inside, best, back


print("cyk defined")

In [ ]:
# Step 4: rebuild and show the best tree
def build_tree(back, words, i, j, A):
    if i == j:
        return (A, [words[i]])
    m, B, C = back[i, j, A]
    return (A, [build_tree(back, words, i, m, B), build_tree(back, words, m + 1, j, C)])


def bracket(tree):
    if isinstance(tree, str):
        return tree
    label, kids = tree
    return "(" + label + " " + " ".join(bracket(k) for k in kids) + ")"


def draw(tree, indent=""):
    if isinstance(tree, str):
        print(indent + tree)
        return
    label, kids = tree
    if len(kids) == 1 and isinstance(kids[0], str):
        print(f"{indent}{label} -> {kids[0]}")
    else:
        print(indent + label)
        for k in kids:
            draw(k, indent + "    ")


print("tree functions defined")

In [ ]:
# Step 5: run CYK on a sentence and print everything
def run(sentence):
    words = sentence.lower().split()
    n = len(words)
    inside, best, back = cyk(words)
    print(f"\nSentence: {sentence}")
    for A in NONTERMINALS:
        print(f"Inside probability of {A}: {inside[0, n - 1, A]:.6f}")
    print(f"P(sentence) = inside probability of S = {inside[0, n - 1, 'S']:.6f}")
    if best[0, n - 1, "S"]:
        print(f"Best parse probability = {best[0, n - 1, 'S']:.6f}")
        tree = build_tree(back, words, 0, n - 1, "S")
        print(bracket(tree))
        draw(tree)
    else:
        print("The grammar cannot generate this sentence")


run("The cat chased the dog")                          # <- change me

In [ ]:
# Ambiguous: the PP can attach to the verb or the noun, so inside (sum of both parses) > best (one parse)
run("The cat chased the dog with the cat")

**Result:** The PCFG was implemented and CYK computed the inside probability of each sentence, plus the most probable parse tree.

**Viva points**
- Inside probability = **sum** over all parses; Viterbi/best parse = **max** over parses.
- CYK needs CNF and takes $O(n^3 |G|)$ time.
- For an ambiguous sentence the inside probability is larger than the probability of the best single parse.

---
# Exercise 8: Text classification with Bag of Words and TF-IDF

**Aim:** Classify text (positive or negative sentiment) using Bag of Words and TF-IDF features.

**Idea**
- **Bag of Words (BoW):** each document becomes a vector of word counts over the vocabulary; order is ignored.
- **TF-IDF:** re-weights the counts so common words matter less.
  $TF(t,d) = \dfrac{\text{count of } t \text{ in } d}{\text{terms in } d}$, $\;IDF(t) = \log \dfrac{\text{documents}}{\text{documents containing } t}$, $\;TFIDF = TF \times IDF$
- **Model:** a single neuron (logistic regression), $p = \sigma(w \cdot x + b)$. This is exactly one `Dense(1, sigmoid)` layer of the TensorFlow version,
  trained here by plain gradient descent.

**Steps (one function each):** 1. `clean` 2. `split_data` 3. `build_vocab` 4. `bow` / `tfidf` 5. `train` / `predict` 6. `train_and_evaluate` 7. `predict_new`

*TensorFlow's `TextVectorization` and `Dense` layers are replaced by the functions below.*

In [ ]:
# Sample data (change me)
import math
import random
import re

random.seed(42)

texts = [
    "I love machine learning and AI", "AI is the future of technology",
    "TensorFlow is a powerful tool", "I enjoy programming with TensorFlow",
    "Deep learning models are amazing", "This library is great and easy to use",
    "The new update is wonderful", "I really like this clean design",
    "Great documentation and helpful examples", "The team built an excellent product",
    "Fast, simple and reliable software", "I am happy with the results",
    "This tool is amazing and easy", "I love the great design",
    "The update is excellent and fast", "Wonderful library and helpful team",
    "I enjoy this reliable software", "Amazing product with great results",
    "I like the simple and clean tool", "The new model is powerful and fast",
    "I hate bugs in my code", "The tool is slow and terrible",
    "This update is awful and broken", "I dislike confusing error messages",
    "The software crashed again, very frustrating", "Worst documentation I have ever read",
    "The app is buggy and unreliable", "I am disappointed with the poor quality",
    "Terrible design and useless features", "This is a horrible and slow product",
    "The library is broken and slow", "I hate this awful update",
    "Poor results and terrible quality", "The software is buggy and frustrating",
    "I dislike the confusing design", "This tool is useless and unreliable",
    "Horrible documentation and awful examples", "The new update is slow and broken",
    "I am disappointed with the terrible app", "The team built a poor product",
]
labels = [1] * 20 + [0] * 20                       # 1 = positive, 0 = negative

STOPWORDS = {"a", "an", "the", "is", "are", "am", "and", "of", "in", "to", "with", "this", "my", "i", "it", "very", "have", "ever"}
new_sentences = ["I love this amazing tool", "This is a terrible broken update"]

In [ ]:
# Step 1: clean the text
def clean(text):
    """lowercase, keep letters only, drop stop words, strip a plural 's' (a tiny lemmatizer)."""
    words = re.findall(r"[a-z]+", text.lower())
    return [w[:-1] if len(w) > 3 and w.endswith("s") and not w.endswith("ss") else w
            for w in words if w not in STOPWORDS]


docs = [clean(t) for t in texts]
print(docs[0], docs[20])

In [ ]:
# Step 2: train / validation split (about 75 / 25, same proportion of each class)
def split_data(docs, labels, val_fraction=0.25):
    pos = [i for i, y in enumerate(labels) if y == 1]
    neg = [i for i, y in enumerate(labels) if y == 0]
    random.shuffle(pos)
    random.shuffle(neg)
    n_pos, n_neg = int(len(pos) * val_fraction), int(len(neg) * val_fraction)
    val_idx = pos[:n_pos] + neg[:n_neg]
    train_idx = pos[n_pos:] + neg[n_neg:]
    return ([docs[i] for i in train_idx], [labels[i] for i in train_idx],
            [docs[i] for i in val_idx], [labels[i] for i in val_idx])


train_docs, y_train, val_docs, y_val = split_data(docs, labels)
print("train:", len(train_docs), "validation:", len(val_docs))

In [ ]:
# Step 3: vocabulary and IDF, built from the TRAINING documents only
def build_vocab(train_docs):
    vocab = sorted({w for d in train_docs for w in d})
    n_docs = len(train_docs)
    idf = {w: math.log(n_docs / sum(1 for d in train_docs if w in d)) for w in vocab}
    return vocab, idf


vocab, idf = build_vocab(train_docs)
print("Vocabulary size:", len(vocab))
print(vocab)

In [ ]:
# Step 4: the two vectorizers
def bow(doc, vocab):
    return [doc.count(w) for w in vocab]                              # raw counts


def tfidf(doc, vocab, idf):
    n = len(doc) or 1
    return [(doc.count(w) / n) * idf[w] for w in vocab]               # TF x IDF


def show_table(title, rows, vocab, fmt):
    """Print only the columns (words) used by the first 4 documents."""
    cols = [j for j in range(len(vocab)) if any(rows[r][j] for r in range(4))][:10]
    print(f"\n{title}  (first 4 training documents)")
    print("       " + " ".join(f"{vocab[j][:9]:>9s}" for j in cols))
    for r in range(4):
        print(f"doc {r}  " + " ".join(fmt.format(rows[r][j]) for j in cols))


show_table("Bag of Words table", [bow(d, vocab) for d in train_docs], vocab, "{:9d}")
show_table("TF-IDF table", [tfidf(d, vocab, idf) for d in train_docs], vocab, "{:9.3f}")

In [ ]:
# Step 5: the model = logistic regression (one sigmoid neuron) trained with gradient descent
def sigmoid(z):
    return 1 / (1 + math.exp(-max(-30, min(30, z))))


def train(X, y, epochs=300, lr=0.5):
    w, b = [0.0] * len(X[0]), 0.0
    for _ in range(epochs):
        for x, target in zip(X, y):
            p = sigmoid(sum(wi * xi for wi, xi in zip(w, x)) + b)
            err = p - target                                          # gradient of the log-loss
            w = [wi - lr * err * xi for wi, xi in zip(w, x)]
            b -= lr * err
    return w, b


def predict(model, x):
    w, b = model
    return sigmoid(sum(wi * xi for wi, xi in zip(w, x)) + b)


def accuracy(model, X, y):
    return sum((predict(model, x) > 0.5) == bool(t) for x, t in zip(X, y)) / len(y)


print("model functions defined")

In [ ]:
# Step 6: build a separate model for Bag of Words and for TF-IDF, train and evaluate
def train_and_evaluate():
    models = {}
    settings = (("Bag of Words", lambda d: bow(d, vocab), 0.3),
                ("TF-IDF", lambda d: tfidf(d, vocab, idf), 1.0))
    for name, vectorize, lr in settings:
        model = train([vectorize(d) for d in train_docs], y_train, lr=lr)
        models[name] = (model, vectorize)
        train_acc = accuracy(model, [vectorize(d) for d in train_docs], y_train)
        val_acc = accuracy(model, [vectorize(d) for d in val_docs], y_val)
        print(f"{name:13s} train accuracy = {train_acc:.2f}   validation accuracy = {val_acc:.2f}")
    return models


models = train_and_evaluate()

In [ ]:
# Step 7: predict new sentences
def predict_new(sentences, models):
    for name, (model, vectorize) in models.items():
        for s in sentences:
            p = predict(model, vectorize(clean(s)))
            print(f"{name:13s} {s!r:40s} -> {'positive' if p > 0.5 else 'negative'} ({p:.2f})")


predict_new(new_sentences, models)

**Result:** Text was classified with Bag of Words and with TF-IDF features; both models were trained, validated and used to predict the sentiment of new sentences.

**Viva points**
- BoW counts words; TF-IDF down-weights words that appear in many documents (IDF is 0 for a word present in every document).
- Neither keeps word order or meaning, which is why word2vec / RNNs came later.
- Fit the vocabulary and IDF on training data only, otherwise validation data leaks into the features.

---
# Exercise 9: word2vec (CBOW and Skip-gram) and semantic similarity

**Aim:** Train a word2vec model and explore the semantic similarity between words.

**Idea:** Words that appear in similar contexts get similar vectors. word2vec is a tiny neural network
(one hidden layer, no activation) whose hidden weights **are** the word vectors.

- **CBOW:** context words (average of their vectors) predict the centre word.
- **Skip-gram:** the centre word predicts each context word.
- Output is a softmax over the vocabulary; training lowers the cross-entropy by gradient descent.
- **Cosine similarity** $= \dfrac{u \cdot v}{\lVert u \rVert\, \lVert v \rVert}$ measures how close two vectors point (1 = same direction).

**Steps (one function each):** 1. `preprocess` 2. `make_examples` 3. `train_word2vec` 4. `cosine` 5. `show_similarity` 6. `most_similar`

*`gensim` is replaced by the small trainer below (full softmax, small corpus).*

In [ ]:
# Sample data (change me)
import math
import random
import re

random.seed(7)

corpus = """
The king rules the kingdom. The queen rules the kingdom. The king wears a golden crown.
The queen wears a golden crown. The king lives in the castle. The queen lives in the castle.
The king rides a white horse. The queen rides a white horse. The king signs the royal law.
The queen signs the royal law.
The dog chases the mouse. The cat chases the mouse. The dog drinks cold water. The cat drinks cold water.
The dog sleeps on the sofa. The cat sleeps on the sofa. The dog eats tasty food. The cat eats tasty food.
The car drives on the road. The bus drives on the road. The car needs more fuel. The bus needs more fuel.
The car has four wheels. The bus has four wheels.
"""
STOPWORDS = {"the", "a", "an", "and", "of", "in", "on", "at", "is"}
DIM, WINDOW = 12, 2                                  # vector size, context window
pairs = [("king", "queen"), ("dog", "cat"), ("car", "bus"), ("king", "dog"), ("dog", "bus")]

In [ ]:
# Step 1: clean the corpus and build the vocabulary
def preprocess(corpus):
    sentences = [[w for w in re.findall(r"[a-z]+", s.lower()) if w not in STOPWORDS]
                 for s in corpus.split(".") if s.strip()]
    vocab = sorted({w for s in sentences for w in s})
    index = {w: i for i, w in enumerate(vocab)}
    return sentences, vocab, index


sentences, vocab, index = preprocess(corpus)
print("vocabulary size:", len(vocab))
print(sentences[:3])

In [ ]:
# Step 2: training examples within the window
def make_examples(mode, sentences, index):
    """CBOW -> ([context ids], centre id).  Skip-gram -> ([centre id], context id)."""
    examples = []
    for s in sentences:
        ids = [index[w] for w in s]
        for pos, centre in enumerate(ids):
            context = [ids[j] for j in range(max(0, pos - WINDOW), min(len(ids), pos + WINDOW + 1)) if j != pos]
            if mode == "cbow":
                examples.append((context, centre))
            else:
                examples.extend(([centre], c) for c in context)
    return examples


print("CBOW example      :", make_examples("cbow", sentences, index)[0])
print("Skip-gram examples:", make_examples("skipgram", sentences, index)[:3])

In [ ]:
# Step 3: train (a small neural network; the rows of W_in are the word vectors)
def train_word2vec(mode, sentences, index, epochs=100, lr=0.05):
    V = len(index)
    W_in = [[random.uniform(-0.5, 0.5) / DIM for _ in range(DIM)] for _ in range(V)]   # word vectors
    W_out = [[0.0] * DIM for _ in range(V)]
    examples = make_examples(mode, sentences, index)
    for _ in range(epochs):
        random.shuffle(examples)
        for inputs, target in examples:
            h = [sum(W_in[i][d] for i in inputs) / len(inputs) for d in range(DIM)]   # hidden layer
            scores = [sum(W_out[v][d] * h[d] for d in range(DIM)) for v in range(V)]
            top = max(scores)
            exps = [math.exp(s - top) for s in scores]
            total = sum(exps)
            grad_h = [0.0] * DIM
            for v in range(V):                                                         # softmax error
                g = exps[v] / total - (1 if v == target else 0)
                for d in range(DIM):
                    grad_h[d] += g * W_out[v][d]
                    W_out[v][d] -= lr * g * h[d]
            for i in inputs:
                for d in range(DIM):
                    W_in[i][d] -= lr * grad_h[d] / len(inputs)
    return W_in


cbow = train_word2vec("cbow", sentences, index)
skip = train_word2vec("skipgram", sentences, index)
print("both models trained")

In [ ]:
# Step 4: cosine similarity
def cosine(u, v):
    dot = sum(a * b for a, b in zip(u, v))
    return dot / (math.sqrt(sum(a * a for a in u)) * math.sqrt(sum(b * b for b in v)))


print("cosine of a vector with itself:", round(cosine(cbow[index["king"]], cbow[index["king"]]), 3))

In [ ]:
# Step 5: similarity of word pairs in both models
def show_similarity(pairs, cbow, skip, index):
    for a, b in pairs:
        for name, model in (("CBOW", cbow), ("Skip Gram", skip)):
            print(f"Cosine similarity between '{a}' and '{b}' - {name} : {cosine(model[index[a]], model[index[b]]):.6f}")


show_similarity(pairs, cbow, skip, index)

In [ ]:
# Step 6: the most similar words to a given word
def most_similar(word, model, vocab, index, topn=4):
    sims = [(w, cosine(model[index[word]], model[index[w]])) for w in vocab if w != word]
    return [(w, round(s, 3)) for w, s in sorted(sims, key=lambda x: -x[1])[:topn]]


for w in ["king", "dog", "car"]:
    print(f"Most similar to '{w}' (Skip Gram):", most_similar(w, skip, vocab, index))

**Result:** CBOW and Skip-gram models were trained and the cosine similarity showed related words (`king`/`queen`, `dog`/`cat`, `car`/`bus`) much closer than unrelated ones.

**Viva points**
- CBOW is faster and good for frequent words; Skip-gram works better for rare words and small data.
- The word vectors are the rows of the input weight matrix; the output matrix is thrown away.
- Real word2vec uses negative sampling or hierarchical softmax instead of a full softmax, to scale to big vocabularies.
- The toy corpus is built so that related words (`king`/`queen`, `dog`/`cat`, `car`/`bus`) share the same contexts. On a real corpus (e.g. Alice in Wonderland) similarities are lower and noisier; the *ordering* related > unrelated is what matters.

---
# Exercise 10: Text generator (language model)

**Aim:** Design a text generator that continues the words given by the user, learned from an available corpus.

**Idea:** Text generation is *language modelling*: learn $P(\text{next word} \mid \text{previous words})$, then repeatedly pick a next word and append it.
The manual does this with an **LSTM**. An LSTM needs a deep-learning library to train, so with the standard library only we use the classic
**n-gram model**, which learns the same thing from counts. The LSTM's three gates are shown separately in the last step.

**Steps (one function each):**
1. `preprocess` (clean the corpus) 2. `build_examples` (predictors X, label y) 3. `train_model` (count next words)
4. `next_word` / `generate_text` 5. `chat_with_generator` (user input) 6. `lstm_step` (the 3 gates)

In [ ]:
# Sample data (change me)
import math
import random
import re
from collections import Counter, defaultdict

random.seed(5)

corpus = """
Natural language processing helps computers understand human language. Natural language processing is used in chatbots,
translation and search engines. Machine learning models learn patterns from large amounts of text.
Machine learning models need clean data and enough examples. Deep learning models read text and predict the next word.
Students study natural language processing in the sixth semester. Students write code to tokenize text and tag words.
A language model predicts the next word from the previous words. A good language model writes fluent text.
Machine translation turns text from one language into another language. Search engines use language models to rank pages.
Chatbots answer questions from users and remember the conversation. Students enjoy building chatbots and translation systems.
Large language models are trained on large amounts of text. Large language models can write code and answer questions.
"""
ORDER = 3                                                   # use up to 2 previous words as context
seeds = ["natural language", "machine learning", "students", "large language"]

In [ ]:
# Step 1: clean the corpus (lowercase, remove punctuation, split into sentences of words)
def preprocess(corpus):
    return [re.findall(r"[a-z]+", s.lower()) for s in corpus.split(".") if s.strip()]


sentences = preprocess(corpus)
print("sentences:", len(sentences), "| words:", sum(map(len, sentences)))

In [ ]:
# Step 2: n-gram sequences = predictors X (previous words) and label y (next word)
def build_examples(sentences, order=ORDER):
    examples = []
    for words in sentences:
        for i in range(1, len(words)):
            for k in range(1, order):                       # context lengths 1 .. order-1
                if i - k >= 0:
                    examples.append((tuple(words[i - k:i]), words[i]))
    return examples


examples = build_examples(sentences)
print("X (previous words)  ->  y (next word)")
for x, y in examples[:6]:
    print("  ", x, "->", y)

In [ ]:
# Step 3: "training" = count how often each next word follows each context
def train_model(examples, sentences):
    model = defaultdict(Counter)
    for x, y in examples:
        model[x][y] += 1
    unigrams = Counter(w for s in sentences for w in s)
    return model, unigrams


model, unigrams = train_model(examples, sentences)
print("contexts learned:", len(model))
print("after 'natural language' comes:", dict(model[("natural", "language")]))

In [ ]:
# Step 4: generation. Sample the next word in proportion to its count.
def next_word(context, model, unigrams, order=ORDER):
    for k in range(order - 1, 0, -1):                       # longest context first, then back off
        key = tuple(context[-k:])
        if key in model:
            options = model[key]
            return random.choices(list(options), weights=list(options.values()))[0]
    return random.choices(list(unigrams), weights=list(unigrams.values()))[0]   # unseen: any word


def generate_text(seed, n_words, model, unigrams):
    words = re.findall(r"[a-z]+", seed.lower())
    if not words:
        return "(please type some words)"
    for _ in range(n_words):
        words.append(next_word(words, model, unigrams))
    return " ".join(words).capitalize()


for seed in seeds:
    print(generate_text(seed, 10, model, unigrams))

In [ ]:
# Step 5: let the user type the seed words (leave empty to stop)
def chat_with_generator(model, unigrams):
    while True:
        seed = input("Seed words (blank to quit): ").strip()
        if not seed:
            break
        n = input("How many words [10]: ").strip()
        print(generate_text(seed, int(n) if n.isdigit() else 10, model, unigrams), "\n")


chat_with_generator(model, unigrams)

In [ ]:
# Step 6: the 3 gates of an LSTM cell (one forward step, random untrained weights: this only shows the data flow)
N_IN, N_HID = 2, 3


def sigmoid(z):
    return 1 / (1 + math.exp(-z))


def new_gate():
    """One gate = a weight row per hidden unit over [x, h_prev], plus a bias."""
    return [([random.uniform(-1, 1) for _ in range(N_IN + N_HID)], 0.0) for _ in range(N_HID)]


def lstm_step(x, h_prev, c_prev, gates):
    z = x + h_prev                                                                   # concatenate input and previous hidden state
    act = lambda g, f: [f(sum(w * v for w, v in zip(row, z)) + b) for row, b in gates[g]]
    f = act("forget", sigmoid)                                                       # 0..1: how much old memory to keep
    i = act("input", sigmoid)                                                        # 0..1: how much new info to write
    g = act("candidate", math.tanh)                                                  # -1..1: the new info itself
    o = act("output", sigmoid)                                                       # 0..1: how much memory to expose
    c = [f[k] * c_prev[k] + i[k] * g[k] for k in range(N_HID)]                       # new cell state
    h = [o[k] * math.tanh(c[k]) for k in range(N_HID)]                               # new hidden state
    return h, c, f, i, o


def run_lstm_demo():
    random.seed(1)
    gates = {name: new_gate() for name in ("forget", "input", "candidate", "output")}
    h, c = [0.0] * N_HID, [0.0] * N_HID
    for t, x in enumerate([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]], 1):
        h, c, f, i, o = lstm_step(x, h, c, gates)
        print(f"t={t} forget={[round(v, 2) for v in f]} input={[round(v, 2) for v in i]} "
              f"output={[round(v, 2) for v in o]} -> h={[round(v, 2) for v in h]}")


run_lstm_demo()

**Result:** A text generator was built from the corpus: given seed words from the user it generated the next words one at a time.

**Viva points**
- Forget gate: what to drop from memory. Input gate: what to add. Output gate: what to show as the next hidden state. All gates are sigmoids (0 to 1).
- An n-gram model only sees the last 1 or 2 words; an LSTM keeps a hidden state, so it can use much longer context.
- Sampling in proportion to counts gives variety; always taking the top word gives repetitive text.

---
# Exercise 11: RNN-based machine translation (English to French)

**Aim:** Build a Recurrent Neural Network based machine translation system in Python.

**Idea:** An **encoder-decoder (seq2seq)** model made of two plain RNNs.

- Encoder RNN reads the English words one by one: $h_t = \tanh(W_{xh}\,x_t + W_{hh}\,h_{t-1} + b_h)$. The last $h$ is a summary of the sentence.
- Decoder RNN starts from that summary and writes French words: $s_t = \tanh(U_{xh}\,y_{t-1} + U_{hh}\,s_{t-1} + b_s)$, then $p_t = \text{softmax}(W_{hy}\,s_t + b_y)$.
- **Teacher forcing:** while training, the decoder is fed the *correct* previous French word. At translation time it is fed its own previous guess.
- Loss = cross-entropy; gradients come from **backpropagation through time (BPTT)**, written by hand below.

**Steps (one function each):**
1. `prepare_data` (vocabularies, ids) 2. `init_params` (weights) 3. `forward` (encoder + decoder)
4. `backward` (BPTT) + `adagrad_update` 5. `train_model` 6. `translate` (greedy decoding)

*TensorFlow/Keras is replaced by plain Python. It works on words (not characters) and uses a very small dataset, so it learns to translate **its training sentences**; it is a demonstration of the architecture, not a general translator.*

In [ ]:
# Sample data (change me)
import math
import random
import re

random.seed(3)

pairs = [("Go.", "Va !"), ("Run!", "Cours !"), ("Who?", "Qui ?"), ("Fire!", "Au feu !"),
         ("Help!", "À l'aide !"), ("Jump.", "Saute."), ("Wait!", "Attends !"), ("Hello!", "Bonjour !"),
         ("I know.", "Je sais."), ("I left.", "Je suis parti."), ("I won!", "J'ai gagné !"),
         ("Get up.", "Lève-toi."), ("Cheers!", "Santé !"), ("Stop!", "Arrête !")]
HIDDEN, EPOCHS, LEARNING_RATE = 24, 120, 0.1

In [ ]:
# Step 1: tokenize, build the two vocabularies, turn words into integer ids
def tokenize(s):
    return re.findall(r"[\w'-]+|[^\w\s]", s.lower())                      # words and punctuation marks


def prepare_data(pairs):
    src_sents = [tokenize(e) for e, _ in pairs]
    tgt_sents = [["<s>"] + tokenize(f) + ["</s>"] for _, f in pairs]      # <s> = start, </s> = end
    s2i = {w: i for i, w in enumerate(sorted({w for s in src_sents for w in s}))}
    t2i = {w: i for i, w in enumerate(sorted({w for s in tgt_sents for w in s}))}
    i2t = {i: w for w, i in t2i.items()}
    data = [([s2i[w] for w in s], [t2i[w] for w in t]) for s, t in zip(src_sents, tgt_sents)]
    return {"s2i": s2i, "t2i": t2i, "i2t": i2t, "data": data}, src_sents, tgt_sents


vocab, src_sents, tgt_sents = prepare_data(pairs)
print(f"English vocabulary = {len(vocab['s2i'])}, French vocabulary = {len(vocab['t2i'])}")
print(src_sents[7], "->", tgt_sents[7])
print("as ids:", vocab["data"][7])

In [ ]:
# Step 2: parameters (the "weights" of the two RNNs). Words are one-hot, so  W @ one_hot(x)  is just column x of W.
def matrix(rows, cols):
    return [[random.uniform(-0.1, 0.1) for _ in range(cols)] for _ in range(rows)]


def init_params(Vs, Vt, H):
    return {"Wxh": matrix(H, Vs), "Whh": matrix(H, H), "bh": matrix(H, 1),       # encoder
            "Uxh": matrix(H, Vt), "Uhh": matrix(H, H), "bs": matrix(H, 1),       # decoder state
            "Why": matrix(Vt, H), "by": matrix(Vt, 1)}                            # decoder output


def zeros_like(params):
    return {k: [[0.0] * len(row) for row in M] for k, M in params.items()}


P = init_params(len(vocab["s2i"]), len(vocab["t2i"]), HIDDEN)
print({name: (len(M), len(M[0])) for name, M in P.items()})

In [ ]:
# Step 3: forward pass = encoder RNN, then decoder RNN (teacher forcing)
def softmax(z):
    top = max(z)
    e = [math.exp(v - top) for v in z]
    total = sum(e)
    return [v / total for v in e]


def rnn_step(Wx, Wh, b, x_id, h_prev):
    H = len(h_prev)
    return [math.tanh(Wx[i][x_id] + sum(Wh[i][k] * h_prev[k] for k in range(H)) + b[i][0]) for i in range(H)]


def output_probs(s, P):
    Vt, H = len(P["by"]), len(s)
    return softmax([sum(P["Why"][v][k] * s[k] for k in range(H)) + P["by"][v][0] for v in range(Vt)])


def forward(src, tgt_in, P):
    H = len(P["bh"])
    h = [0.0] * H
    hs = [h]
    for x in src:                                                          # encoder
        h = rnn_step(P["Wxh"], P["Whh"], P["bh"], x, h)
        hs.append(h)
    s, ss, probs = h, [h], []
    for y in tgt_in:                                                       # decoder (starts from the encoder summary)
        s = rnn_step(P["Uxh"], P["Uhh"], P["bs"], y, s)
        ss.append(s)
        probs.append(output_probs(s, P))
    return hs, ss, probs


src_ids, tgt_ids = vocab["data"][7]
_, _, probs = forward(src_ids, tgt_ids[:-1], P)
print("untrained model, loss for one sentence:",
      round(-sum(math.log(p[t]) for p, t in zip(probs, tgt_ids[1:])), 3))

In [ ]:
# Step 4: backward pass (backpropagation through time) and the weight update
def backward(src, tgt, G, P):
    """Adds the gradients into G and returns the loss of this sentence pair."""
    H, Vt = len(P["bh"]), len(P["by"])
    tgt_in, tgt_out = tgt[:-1], tgt[1:]                                    # teacher forcing: input shifted by one word
    hs, ss, probs = forward(src, tgt_in, P)
    loss, ds_next = 0.0, [0.0] * H
    for t in reversed(range(len(tgt_out))):                                # decoder, last word to first
        dlogits = probs[t][:]
        dlogits[tgt_out[t]] -= 1                                           # softmax + cross-entropy gradient
        loss -= math.log(probs[t][tgt_out[t]])
        s, s_prev = ss[t + 1], ss[t]
        ds = [ds_next[k] + sum(P["Why"][v][k] * dlogits[v] for v in range(Vt)) for k in range(H)]
        for v in range(Vt):
            G["by"][v][0] += dlogits[v]
            for k in range(H):
                G["Why"][v][k] += dlogits[v] * s[k]
        dpre = [ds[i] * (1 - s[i] ** 2) for i in range(H)]                 # through tanh
        for i in range(H):
            G["Uxh"][i][tgt_in[t]] += dpre[i]
            G["bs"][i][0] += dpre[i]
            for k in range(H):
                G["Uhh"][i][k] += dpre[i] * s_prev[k]
        ds_next = [sum(P["Uhh"][i][k] * dpre[i] for i in range(H)) for k in range(H)]
    dh_next = ds_next                                                      # gradient flows into the encoder's last state
    for t in reversed(range(len(src))):                                    # encoder, last word to first
        h, h_prev = hs[t + 1], hs[t]
        dpre = [dh_next[i] * (1 - h[i] ** 2) for i in range(H)]
        for i in range(H):
            G["Wxh"][i][src[t]] += dpre[i]
            G["bh"][i][0] += dpre[i]
            for k in range(H):
                G["Whh"][i][k] += dpre[i] * h_prev[k]
        dh_next = [sum(P["Whh"][i][k] * dpre[i] for i in range(H)) for k in range(H)]
    return loss


def adagrad_update(P, G, cache, lr):
    for name, M in P.items():
        for r in range(len(M)):
            for c in range(len(M[r])):
                g = max(-5.0, min(5.0, G[name][r][c]))                     # clip exploding gradients
                cache[name][r][c] += g * g
                M[r][c] -= lr * g / (math.sqrt(cache[name][r][c]) + 1e-8)


print("backward + update defined")

In [ ]:
# Step 5: train for a number of epochs
def train_model(P, data, epochs, lr):
    cache = zeros_like(P)
    for epoch in range(1, epochs + 1):
        random.shuffle(data)
        total = 0.0
        for src, tgt in data:
            G = zeros_like(P)
            total += backward(src, tgt, G, P)
            adagrad_update(P, G, cache, lr)
        if epoch % 20 == 0 or epoch == 1:
            print(f"epoch {epoch:3d}   average loss per sentence = {total / len(data):.4f}")


train_model(P, vocab["data"], EPOCHS, LEARNING_RATE)

In [ ]:
# Step 6: translate = encode the English words, then decode greedily (most probable French word each step)
def translate(sentence, P, vocab, max_len=10):
    s2i, t2i, i2t = vocab["s2i"], vocab["t2i"], vocab["i2t"]
    src = [s2i[w] for w in tokenize(sentence) if w in s2i]                 # unknown words are skipped
    h = [0.0] * len(P["bh"])
    for x in src:
        h = rnn_step(P["Wxh"], P["Whh"], P["bh"], x, h)
    s, y, out = h, t2i["<s>"], []
    for _ in range(max_len):
        s = rnn_step(P["Uxh"], P["Uhh"], P["bs"], y, s)
        probs = output_probs(s, P)
        y = max(range(len(probs)), key=lambda v: probs[v])                 # most probable next word
        if i2t[y] == "</s>":
            break
        out.append(i2t[y])
    return " ".join(out)


print("English to French Translation")
for sentence, expected in pairs[:8] + pairs[-3:]:                          # <- change me
    print(f"Input sentence: {sentence}\nDecoded sentence: {translate(sentence, P, vocab)}   (expected: {expected})\n")

**Result:** An RNN encoder-decoder was built and trained with backpropagation through time; it translates the English sentences into French.

**Viva points**
- The encoder compresses the whole sentence into one vector; the decoder generates the output word by word from it (the "bottleneck" that attention later fixed).
- Plain RNNs suffer from vanishing/exploding gradients on long sentences (here: gradient clipping); LSTM/GRU use gates to help.
- Teacher forcing speeds up training but creates a train/test mismatch (at test time the model sees its own mistakes).

---
# Exercise 12: Chatbot using a Large Language Model (LLM)

**Aim:** Develop a chatbot using a Large Language Model.

**Idea:** An LLM is called over an HTTP API: we send the **system prompt** (the bot's role) plus the **conversation so far**,
and get back the next reply. The model itself has no memory, so the *chatbot's memory is the `history` list we send every time*.

**Steps (one function each):**
1. `llm_reply` (call the LLM with `urllib`) 2. `offline_reply` (fallback bot, used when there is no key or internet)
3. `chat` (history + memory + trimming) 4. `demo_chat` (sample questions) 5. `interactive_chat` (you type)

**To use the real LLM:** set the environment variable `ANTHROPIC_API_KEY` before starting Jupyter
(PowerShell: `$env:ANTHROPIC_API_KEY = "sk-ant-..."`), or type the key in the first cell.

In [ ]:
# Sample data and settings (change me)
import json
import os
import re
import urllib.error
import urllib.request

MODEL = "claude-sonnet-5-5"
API_URL = "https://api.anthropic.com/v1/messages"
API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")            # or: API_KEY = "sk-ant-..."
MAX_TURNS = 10                                                # memory: keep the last 10 question/answer pairs

SYSTEM = ("You are a friendly assistant for a perfume shop. We sell floral, woody, citrus, oriental and "
          "fresh fragrances for men, women and unisex. Be concise and helpful; if you do not know "
          "something about the shop, say so.")

CATALOG = {
    "floral": "Floral: rose, jasmine and lily fragrances, soft and romantic. Popular for women and unisex.",
    "woody": "Woody: sandalwood, cedar and oud notes, warm and deep. Popular for men and unisex.",
    "citrus": "Citrus: lemon, bergamot and orange, bright and energetic. Great for daytime and summer.",
    "oriental": "Oriental: amber, vanilla and spice, rich and long-lasting. Best for evenings.",
    "fresh": "Fresh: sea breeze, mint and green tea, light and clean. Good for everyday wear.",
}

history = []                                                  # [{"role": "user"/"assistant", "content": "..."}]
memory = {"topic": None, "name": None}                        # used by the offline bot
questions = ["What types of perfumes do you sell?", "Tell me about woody ones", "Tell me more about it"]

In [ ]:
# Step 1: call the LLM (standard library only)
def llm_reply(messages):
    """Returns the reply text, or None if the call fails."""
    body = json.dumps({"model": MODEL, "max_tokens": 500, "system": SYSTEM, "messages": messages}).encode()
    request = urllib.request.Request(API_URL, data=body, headers={
        "x-api-key": API_KEY, "anthropic-version": "2023-06-01", "content-type": "application/json"})
    try:
        with urllib.request.urlopen(request, timeout=30) as response:
            return json.load(response)["content"][0]["text"]
    except (urllib.error.URLError, KeyError, ValueError) as error:
        print(f"[LLM call failed ({error}); using the offline bot]")
        return None


print("Mode:", "LLM API" if API_KEY else "offline bot (no ANTHROPIC_API_KEY set)")

In [ ]:
# Step 2: offline fallback bot = keyword matching plus a small memory (last topic, user's name)
def offline_reply(message):
    text = message.lower()
    found = re.search(r"my name is (\w+)", text)
    if found:
        memory["name"] = found.group(1).capitalize()
        return f"Nice to meet you, {memory['name']}! How can I help with perfumes today?"
    topics = [t for t in CATALOG if t in text]
    if topics:
        memory["topic"] = topics[0]
        return CATALOG[topics[0]]
    if re.search(r"\b(types|kinds|sell|range|collection)\b", text):
        return "We sell " + ", ".join(CATALOG) + " perfumes for men, women and unisex."
    if re.search(r"\b(price|cost|cheap|expensive)\b", text):
        return "I do not have the price list here, please check with the shop counter."
    if re.search(r"\b(hi|hello|hey)\b", text):
        return f"Hello{' ' + memory['name'] if memory['name'] else ''}! Ask me about our perfumes."
    if re.search(r"\b(it|that|them|more|those)\b", text) and memory["topic"]:    # uses memory of the last topic
        return CATALOG[memory["topic"]] + " Would you like it for men, women or unisex?"
    if re.search(r"\b(bye|thanks|thank)\b", text):
        return "You are welcome! Have a nice day."
    return "Sorry, I do not know about that. I can tell you about our perfume types."


print(offline_reply("Hello"))

In [ ]:
# Step 3: one chat turn = add the message to the history, trim it, get a reply, store the reply
def chat(message):
    history.append({"role": "user", "content": message})
    del history[:max(0, len(history) - 2 * MAX_TURNS)]                      # trim old context
    reply = llm_reply(history) if API_KEY else None
    if reply is None:
        reply = offline_reply(message)
    history.append({"role": "assistant", "content": reply})
    return reply


print(chat("What types of perfumes do you sell?"))

In [ ]:
# Step 4: run a few sample questions in one conversation
def demo_chat(questions):
    for q in questions:
        print(f"You: {q}\nBot: {chat(q)}\n")


history.clear()
memory.update(topic=None, name=None)
demo_chat(questions)

In [ ]:
# Step 5: chat yourself (type quit to stop)
def interactive_chat():
    while (message := input("You: ").strip()).lower() not in {"quit", "exit", ""}:
        print("Bot:", chat(message), "\n")
    print(f"Conversation length kept in memory: {len(history)} messages")


interactive_chat()

**Result:** A chatbot was built around an LLM: user messages and the running history are sent to the model, and the reply keeps the conversation context. An offline bot provides the same behaviour when no API is available.

**Viva points**
- The system prompt sets role and tone; the `history` list gives multi-turn memory; trimming it keeps the request within the context window.
- Fine-tuning changes the model's weights; prompting (what we do here) changes only the input. For a shop bot, prompting plus the product list is usually enough.
- Do not hard-code API keys in notebooks; read them from environment variables.